In [ ]:
!uv pip install dask dask-geopandas datashader scikit-learn matplotlib-inline

# AEF-BNG: Local Workflow Example

***Assumes you've ran the example CLI code for London.***

```bash
aef-bng process \                  
    --year 2025 \                             
    --bounds 508848 163362 553002 196746 \    
    --workers 8 \
    --output ./london_aef
```

This notebook demonstrates working with the GeoParquet output from `aef-bng`:

1. Reading the output files
2. Inspecting the data
3. Dequantising embeddings to float values
4. Visualising embedding bands as an RGB composite using `geopandas`
5. RGB + PCA composite with `datashader`

## 0. Read example and plot

Looking at Vauxhall Brdige in London to get an understanding of cell size scale.

In [ ]:
import contextily

from aef_bng.types import BoundingBox
from aef_bng.writer import read_output

folder = "../london_aef/2025"

ax = read_output(
    "../london_aef/2025",
    bbox=(
        BoundingBox(-14824.0062, 6707404.1711, -13687.0054, 6708469.5122)
        .reproject(3857, 27700)
        .as_tuple()
    ),
).plot(fc="None", lw=0.2)

contextily.add_basemap(ax=ax, crs=27700)

You can also use `geoparquet-io` for checking metadata, statistics etc..

In [ ]:
import geoparquet_io as gpio

gpio.read(folder).info()

## 1. Read a subset of the extacted dataset

The pipeline writes GeoParquet files with 10m BNG polygon geometry, a `bng_ref` column, `year`, and 64 int8 embedding bands (`A00`..`A63`).

In [ ]:
bbox = BoundingBox(527693, 177259, 532584, 181344).as_tuple()

gdf = read_output("../london_aef/2025", bbox=bbox)
print(f"Rows:     {len(gdf):,}")
print(f"CRS:      {gdf.crs.to_epsg()}")
print(f"Geom type: {gdf.geometry.geom_type.iloc[0]}")
print(f"Columns:  {list(gdf.columns[:5])} ... {list(gdf.columns[-4:])}")
gdf.head(3)

Check cell bounds and area.

In [ ]:
cell = gdf.geometry.iloc[0]
print(f"Cell bounds: {cell.bounds}")
print(f"Cell area:   {cell.area} m²  (expected: 100)")

Verify the assigned British National Grid reference is correct.

In [ ]:
# Verify BNG references match geometry
from osbng import BNGReference

sample = gdf.iloc[0]
ref = BNGReference(sample.bng_ref)
ref_poly = ref.bng_to_grid_geom()

print(f"bng_ref:        {sample.bng_ref}")
print(f"Geometry bounds: {sample.geometry.bounds}")
print(f"osbng bounds:    {ref_poly.bounds}")
print(f"Geometry match:           {sample.geometry.equals(ref_poly)}")

## 2. Dequantise embeddings

Raw AEF embeddings are stored as int8 values in [-127, 127] (with -128 as nodata).
The dequantisation formula maps these to float32 in [-1, 1]:

```
dequantised = (value / 127.5)² × sign(value)
```

The `dequantise_dataframe` function applies this to all 64 band columns at once.

In [ ]:
from aef_bng.dequantise import dequantise_dataframe

gdf_dq = dequantise_dataframe(gdf)

print("Before dequantisation:")
print(f"  A00 dtype: {gdf['A00'].dtype}, range: [{gdf['A00'].min()}, {gdf['A00'].max()}]")
print()
print("After dequantisation:")
print(
    f"  A00 dtype: {gdf_dq['A00'].dtype}, range: [{gdf_dq['A00'].min():.4f}, {gdf_dq['A00'].max():.4f}]"  # noqa: E501
)

gdf_dq[["bng_ref", "year", "A00", "A01", "A02"]].head()

## 3. RGB visualisation from embedding bands

AEF embeddings encode spectral and spatial features into 64 learned dimensions.
While they don't correspond to specific wavelengths, assigning three bands to
RGB channels gives a useful false-colour visualisation of the landscape.

Different band combinations highlight different landscape features.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def plot_embedding_rgb(
    gdf,
    r_band="A00",
    g_band="A01",
    b_band="A02",
    title=None,
    figsize=(12, 10),
):
    """Plot an RGB composite from three AEF embedding bands.

    Renders the actual 10m BNG polygon geometry, coloured by normalising
    the selected bands to [0, 1] and mapping to RGB.
    """
    r = gdf[r_band].to_numpy().astype(np.float64)
    g = gdf[g_band].to_numpy().astype(np.float64)
    b = gdf[b_band].to_numpy().astype(np.float64)

    def norm(arr):
        lo, hi = np.nanpercentile(arr, [2, 98])
        return np.clip((arr - lo) / (hi - lo + 1e-10), 0, 1)

    rgb = np.stack([norm(r), norm(g), norm(b)], axis=-1)

    fig, ax = plt.subplots(1, 1, figsize=figsize)
    gdf.plot(ax=ax, color=[tuple(c) for c in rgb], linewidth=0, antialiased=True)
    ax.set_axis_off()
    ax.set_aspect("equal")
    ax.set_title(title or f"AEF Embedding RGB: R={r_band}, G={g_band}, B={b_band}", fontsize=18)
    plt.tight_layout()
    return fig, ax

Visualise for a sample area using the first 3 bands.


In [ ]:
fig, ax = plot_embedding_rgb(
    gdf_dq,
    r_band="A00",
    g_band="A01",
    b_band="A02",
    figsize=(14, 14),
)
plt.show()

del gdf, gdf_dq

## 4. Datashader RGB + PCA composite

For larger datasets `geopandas` and `matplotlib` struggle to plot. Instead, we'll use
[`datashader`](https://datashader.org/)
to rasterise millions of records directly to a pixel grid.

Two composites are shown side by side:

- **Direct RGB**: bands A00, A01, A02 mapped to RGB after 2–98 percentile
  stretch. Equivalent to the plot above, but rendered as a raster.
- **PCA RGB**: all 64 bands compressed to 3 principal components, each mapped to
  a colour channel. PCA extracts the dominant axes of spectral variation across the
  full embedding space, so the composite tends to show more landscape contrast than
  any fixed band selection.

The canvas is sized to preserve the 1:1 BNG metre-per-pixel aspect ratio as
a fixed square canvas on a non-square extent causes geometric distortion.

In [ ]:
from pathlib import Path

import dask.dataframe as dd
import dask_geopandas
import datashader as ds
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import shapely
from sklearn.decomposition import PCA

from aef_bng.constants import AEF_BAND_NAMES
from aef_bng.dequantise import dequantise

PARQUET_DIR = Path(folder)
OUTPUT_JPG = PARQUET_DIR.parent / "london-rgb-pca-composite.jpg"

SAMPLE_FRAC = 0.25

MAX_CANVAS_PX = 3000

STRETCH_LO = 2.0
STRETCH_HI = 98.0

RGB_BANDS = ("A00", "A01", "A02")

LOAD_COLS = ["easting", "northing", *AEF_BAND_NAMES]

print(f"Loading: {PARQUET_DIR}")
ddf = dd.read_parquet(str(PARQUET_DIR), columns=LOAD_COLS)
print(f"  {ddf.npartitions} partitions")

print("Computing spatial extent...")
e_min, e_max, n_min, n_max = dd.compute(
    ddf["easting"].min(),
    ddf["easting"].max(),
    ddf["northing"].min(),
    ddf["northing"].max(),
)
x_range = (int(e_min), int(e_max) + 10)
y_range = (int(n_min), int(n_max) + 10)
print(f"  Easting:  {x_range[0]:,} - {x_range[1]:,} m")
print(f"  Northing: {y_range[0]:,} - {y_range[1]:,} m")

e_span = x_range[1] - x_range[0]
n_span = y_range[1] - y_range[0]

if n_span >= e_span:
    canvas_h = MAX_CANVAS_PX
    canvas_w = max(1, int(MAX_CANVAS_PX * e_span / n_span))
else:
    canvas_w = MAX_CANVAS_PX
    canvas_h = max(1, int(MAX_CANVAS_PX * n_span / e_span))

print(f"  Canvas: {canvas_w} x {canvas_h} px  ({e_span / canvas_w:.1f} m/px)")

print("PCA fit...")
sample = ddf.compute()
print(f"  Sample size: {len(sample):,}")
raw_sample = sample[AEF_BAND_NAMES].to_numpy()
deq_sample = dequantise(raw_sample)
valid = ~np.any(np.isnan(deq_sample), axis=1)
print(f"  Valid rows (non-nodata): {valid.sum():,}")
pca = PCA(n_components=3, whiten=True, random_state=42)
pca.fit(deq_sample[valid])
explained = pca.explained_variance_ratio_ * 100
print(
    f"  Explained variance: "
    f"PC1={explained[0]:.1f}%  PC2={explained[1]:.1f}%  PC3={explained[2]:.1f}%  "
    f"(total={sum(explained):.1f}%)"
)


def _build_geometry(eastings: np.ndarray, northings: np.ndarray) -> gpd.GeoSeries:
    """Build a GeoSeries of 10m x 10m BNG cell polygons from lower-left coordinates."""
    e = eastings.astype(np.float64)
    n = northings.astype(np.float64)
    return gpd.GeoSeries(shapely.box(e, n, e + 10.0, n + 10.0))


_meta_pca = gpd.GeoDataFrame(
    {
        "pc1": pd.Series(dtype="float64"),
        "pc2": pd.Series(dtype="float64"),
        "pc3": pd.Series(dtype="float64"),
        "geometry": gpd.GeoSeries(dtype="geometry"),
    }
)


def _project_pca(df: pd.DataFrame) -> gpd.GeoDataFrame:
    """Dequantise and project a partition's band columns onto PCA components."""
    raw = df[AEF_BAND_NAMES].to_numpy()
    deq = dequantise(raw)
    valid_mask = ~np.any(np.isnan(deq), axis=1)

    pc = np.full((len(df), 3), np.nan)
    if valid_mask.any():
        pc[valid_mask] = pca.transform(deq[valid_mask])

    geom = _build_geometry(df["easting"].to_numpy(), df["northing"].to_numpy())
    return gpd.GeoDataFrame(
        {
            "pc1": pc[:, 0],
            "pc2": pc[:, 1],
            "pc3": pc[:, 2],
        },
        geometry=geom.values,
        index=df.index,
    )


_meta_rgb = gpd.GeoDataFrame(
    {
        "r": pd.Series(dtype="float64"),
        "g": pd.Series(dtype="float64"),
        "b": pd.Series(dtype="float64"),
        "geometry": gpd.GeoSeries(dtype="geometry"),
    }
)


def _project_rgb(df: pd.DataFrame) -> gpd.GeoDataFrame:
    """Dequantise RGB_BANDS for a direct RGB composite."""
    raw = df[list(RGB_BANDS)].to_numpy()
    deq = dequantise(raw)  # nodata (-128) → NaN

    geom = _build_geometry(df["easting"].to_numpy(), df["northing"].to_numpy())
    return gpd.GeoDataFrame(
        {
            "r": deq[:, 0],
            "g": deq[:, 1],
            "b": deq[:, 2],
        },
        geometry=geom.values,
        index=df.index,
    )


pca_ddf = dask_geopandas.from_dask_dataframe(
    ddf.map_partitions(_project_pca, meta=_meta_pca), geometry="geometry"
)
rgb_ddf = dask_geopandas.from_dask_dataframe(
    ddf.map_partitions(_project_rgb, meta=_meta_rgb), geometry="geometry"
)

_full_extent = gpd.GeoSeries(
    [shapely.box(x_range[0], y_range[0], x_range[1], y_range[1])] * ddf.npartitions,
    crs=27700,
)

# hacky way of setting dask-geopandas spatial partitions as we're not needing them for anything
pca_ddf.spatial_partitions = _full_extent
rgb_ddf.spatial_partitions = _full_extent

print(f"Rasterising to {canvas_w} x {canvas_h} px canvas...")

cvs = ds.Canvas(
    plot_width=canvas_w,
    plot_height=canvas_h,
    x_range=x_range,
    y_range=y_range,
)

pca_raw: dict[str, np.ndarray] = {}
for col, label in [("pc1", "R"), ("pc2", "G"), ("pc3", "B")]:
    print(f"  PCA  {col} → {label}...")
    pca_raw[col] = cvs.polygons(pca_ddf, geometry="geometry", agg=ds.mean(col)).values

rgb_raw: dict[str, np.ndarray] = {}
for col, label in [("r", "R"), ("g", "G"), ("b", "B")]:
    print(f"  RGB  {col} → {label}...")
    rgb_raw[col] = cvs.polygons(rgb_ddf, geometry="geometry", agg=ds.mean(col)).values


def _stretch(arr: np.ndarray) -> np.ndarray:
    """Percentile-stretch a 2D float array to uint8; nodata pixels → 0."""
    valid_vals = arr[~np.isnan(arr)]
    if len(valid_vals) == 0:
        return np.zeros_like(arr, dtype=np.uint8)
    p_lo, p_hi = np.percentile(valid_vals, [STRETCH_LO, STRETCH_HI])
    clipped = np.clip((arr - p_lo) / (p_hi - p_lo + 1e-9), 0.0, 1.0)
    clipped[np.isnan(arr)] = 0.0
    return (clipped * 255).astype(np.uint8)


pca_img = np.stack(
    [_stretch(pca_raw["pc1"]), _stretch(pca_raw["pc2"]), _stretch(pca_raw["pc3"])],
    axis=-1,
)[::-1]

rgb_img = np.stack(
    [_stretch(rgb_raw["r"]), _stretch(rgb_raw["g"]), _stretch(rgb_raw["b"])],
    axis=-1,
)[::-1]


print("Composing figure...")
panel_w_in = 12.0
panel_h_in = panel_w_in * canvas_h / canvas_w

fig, axes = plt.subplots(
    1,
    2,
    figsize=(panel_w_in * 2, panel_h_in + 1.2),
)

axes[0].imshow(rgb_img, interpolation="nearest")
axes[0].set_axis_off()
axes[0].set_aspect("equal")
axes[0].set_title(
    f"AEF Embeddings: London, UK (2025)\nRGB: R={RGB_BANDS[0]}, G={RGB_BANDS[1]}, B={RGB_BANDS[2]}",
    fontsize=18,
)

axes[1].imshow(pca_img, interpolation="nearest")
axes[1].set_axis_off()
axes[1].set_aspect("equal")
axes[1].set_title(
    f"PCA RGB (64 \u2192 3 components)\nExplained variance: {sum(explained):.1f}%",
    fontsize=18,
)

plt.tight_layout()
fig.savefig(str(OUTPUT_JPG), dpi=100, bbox_inches="tight")
print(f"\nSaved: {OUTPUT_JPG}")

In [ ]:
!uv pip uninstall dask dask-geopandas datashader scikit-learn matplotlib-inline